# GPU Ray-Traced Urban Shadow Extraction

**3-bit encoded shadow mapping** using **DTM + DSM** on CUDA.

This notebook contains the **full production pipeline** for the shadow rasters used in the project.

## What this notebook does

1. **Setup**  
   Imports, CUDA kernel definition, configuration, helper functions.

2. **Load and validate inputs**  
   Reads the DTM and DSM, checks that they are aligned on the same grid, and stores metadata about the input rasters.

3. **Generate timestamps and save the run configuration**  
   Builds the list of timestamps to process and writes a full configuration snapshot so the run can be reproduced later.

4. **Run the GPU ray tracing**  
   Processes all timestamps tile by tile and writes the output GeoTIFFs.

5. **Run integrity checks and diagnostics**  
   Verifies that only valid encoded values are present, produces QA tables, and compares the two output heights.

## Shadow logic implemented here

For each analysis height `h`, the raster stores three logically separate pieces of information:

- **`G_h`** = ground / pedestrian shadow from the main DTM+DSM ray trace  
- **`S_h`** = DSM surface shadow, filtered so it is only kept where the local object is above the selected height  
- **`M_h`** = object mask, i.e. pixels where `DSM > DTM + h + ε`

The encoded value is:

```text
value_h = 1*G_h + 2*S_h + 4*M_h
```

This means the raster is **not** a simple class map. It is a compact 3-bit product from which multiple analytical layers can be extracted later.

## 3-bit encoding

| Value | M_h | S_h | G_h | Meaning |
|-------|-----|-----|-----|---------|
| 0 | 0 | 0 | 0 | Clear sun |
| 1 | 0 | 0 | 1 | Ground / pedestrian shadow only |
| 4 | 1 | 0 | 0 | Object present above the analysis height, but no shadow on its upper surface |
| 5 | 1 | 0 | 1 | Object present above the analysis height and shadow in the main DTM+DSM model |
| 6 | 1 | 1 | 0 | Object present above the analysis height and DSM surface shadow only |
| 7 | 1 | 1 | 1 | Full overlap: object mask + DSM surface shadow + DTM+DSM shadow |
| 255 | — | — | — | Nodata |

Values **2** and **3** are intentionally impossible, because `S_h` is only defined where `M_h = 1`. The QA block later in the notebook checks this explicitly.

## How to read the outputs

Some common derived layers are:

- **Total pedestrian shadow**: `((value & 1) != 0) OR ((value & 4) != 0)`
- **DSM shadow above high objects only**: `((value & 2) != 0)`
- **Object mask above height h**: `((value & 4) != 0)`
- **Hybrid visualization layer**: `((value & 2) != 0) OR (((value & 1) != 0) AND ((value & 4) == 0))`

The code cells below generate the encoded master rasters; the aggregation and downstream interpretation happen later in separate scripts.


## 1. Imports

This section imports all libraries required by the pipeline.

Main dependency groups:

- **Numerical processing**: `numpy`, `cupy`
- **Raster I/O**: `rasterio`
- **Solar position**: `pvlib`
- **Configuration and metadata**: `dataclasses`, `json`, `csv`
- **Utilities**: `datetime`, `pathlib`, `glob`, `pandas`

Run this cell once at the beginning of the notebook.


In [1]:
from __future__ import annotations

import csv
import glob
import json
import math
import os
import time
from dataclasses import dataclass, field, asdict
from datetime import date, datetime, time as dt_time, timedelta
from pathlib import Path

# NVIDIA DLL path for NVRTC (Windows pip installs)
import importlib.util
_nvidia_nvrtc = importlib.util.find_spec("nvidia.cuda_nvrtc")
if (
    hasattr(os, "add_dll_directory")
    and _nvidia_nvrtc
    and _nvidia_nvrtc.submodule_search_locations
):
    _nvrtc_bin = os.path.join(list(_nvidia_nvrtc.submodule_search_locations)[0], "bin")
    if os.path.isdir(_nvrtc_bin):
        os.add_dll_directory(_nvrtc_bin)

import cupy as cp
import numpy as np
import pandas as pd
import rasterio
from pvlib.solarposition import get_solarposition

print(f"CuPy: {cp.__version__}")
props = cp.cuda.runtime.getDeviceProperties(0)
gpu_name = props["name"].decode() if isinstance(props["name"], bytes) else props["name"]
print(f"GPU: {gpu_name}")

c:\Users\leona\AppData\Local\Programs\Python\Python312\Lib\site-packages\cupy\_environment.py:275: UserWarning: CUDA path could not be detected. Set CUDA_PATH environment variable if CuPy fails to load.
  warnings.warn(


CuPy: 14.0.1
GPU: NVIDIA GeForce RTX 4070 Laptop GPU


## 2. CUDA Kernel

This cell defines the GPU kernel used for the shadow calculation.

### Conceptual model

Each output pixel stores three components:

1. **`M_h` (static object mask)**  
   `DSM > DTM + h + ε`  
   This identifies pixels where the local DSM surface is above the selected analysis height.

2. **`G_h` (main DTM+DSM shadow)**  
   A ray starts from `DTM + h` and is tested against the DSM along the sun direction.  
   This is the physically relevant shadow layer for ground / pedestrian analysis.

3. **`S` (DSM surface shadow)**  
   A ray starts from the DSM surface itself. This is calculated once and later filtered by `M_h` to obtain `S_h`.

### Why this structure matters

- `G_h` answers: **is a point at height h above the ground in shadow?**
- `M_h` answers: **is there already an object above height h at this pixel?**
- `S_h` answers: **is the upper surface of that high object in shadow?**

Keeping these three pieces separate makes it possible to derive multiple analytical layers later without recomputing the ray trace.

### Performance notes

The kernel includes several optimizations:

- conditional DSM surface ray computation
- adaptive interpolation (bilinear on smooth surfaces, nearest near sharp edges)
- early termination using the local maximum DSM value
- sub-pixel stepping for more stable shadow edges
- optional dual-height tracing in a single pass


In [2]:
_KERNEL_CODE = r"""

extern "C" __global__
void shadow_raytrace_dtm_dsm_bits(
    const float* __restrict__ dtm,
    const float* __restrict__ dsm,
    unsigned char* __restrict__ encoded_out,
    unsigned char* __restrict__ encoded_out_secondary,
    const int rows,
    const int cols,
    const float step_col,
    const float step_row,
    const float z_rise_per_step,
    const int max_march_steps,
    const float observer_height,
    const float ground_ray_offset,
    const float observer_height_secondary,
    const float ground_ray_offset_secondary,
    const float dsm_surface_offset,
    const float object_mask_epsilon,
    const float max_dsm_z,
    const float discontinuity_threshold,
    const int dual_trace
) {
    const int col = blockIdx.x * blockDim.x + threadIdx.x;
    const int row = blockIdx.y * blockDim.y + threadIdx.y;
    if (row >= rows || col >= cols) return;

    const int idx = row * cols + col;
    const float ground_z = dtm[idx];
    const float surface_start_z = dsm[idx];

    if (ground_z != ground_z || surface_start_z != surface_start_z) {
        encoded_out[idx] = 255;
        if (dual_trace) encoded_out_secondary[idx] = 255;
        return;
    }

    /* Per-height object masks: DSM surface above the analysis height */
    const int mask_primary = (surface_start_z > (ground_z + observer_height + object_mask_epsilon)) ? 1 : 0;
    const int mask_secondary = (dual_trace && surface_start_z > (ground_z + observer_height_secondary + object_mask_epsilon)) ? 1 : 0;

    /* Ray origins for ground / pedestrian shadow */
    const float z_base_primary = ground_z + observer_height + ground_ray_offset;
    const float z_base_secondary = ground_z + observer_height_secondary + ground_ray_offset_secondary;

    /* Shared DSM-surface ray: traced once, then filtered with M_h outside the kernel */
    const float z_base_dsm = surface_start_z + dsm_surface_offset;
    const int need_dsm = mask_primary || (dual_trace && mask_secondary);

    int hit_primary = 0;
    int hit_secondary = 0;
    int hit_dsm = 0;

    for (int step = 1; step <= max_march_steps; step++) {
        const float px = (float)col + step_col * (float)step;
        const float py = (float)row + step_row * (float)step;
        const float ray_z_primary = z_base_primary + z_rise_per_step * (float)step;
        const float ray_z_secondary = z_base_secondary + z_rise_per_step * (float)step;
        const float ray_z_dsm = z_base_dsm + z_rise_per_step * (float)step;

        float lowest_unresolved = 1e30f;
        if (!hit_primary) lowest_unresolved = fminf(lowest_unresolved, ray_z_primary);
        if (dual_trace && !hit_secondary) lowest_unresolved = fminf(lowest_unresolved, ray_z_secondary);
        if (need_dsm && !hit_dsm) lowest_unresolved = fminf(lowest_unresolved, ray_z_dsm);
        if (lowest_unresolved > max_dsm_z) {
            break;
        }

        if (px < 0.0f || py < 0.0f || px >= (float)cols || py >= (float)rows) {
            break;
        }

        int ix = (int)floorf(px);
        int iy = (int)floorf(py);
        ix = max(0, min(ix, cols - 1));
        iy = max(0, min(iy, rows - 1));
        const int ix1 = min(ix + 1, cols - 1);
        const int iy1 = min(iy + 1, rows - 1);

        const float fx = px - (float)ix;
        const float fy = py - (float)iy;

        const float v00 = dsm[iy  * cols + ix ];
        const float v10 = dsm[iy  * cols + ix1];
        const float v01 = dsm[iy1 * cols + ix ];
        const float v11 = dsm[iy1 * cols + ix1];

        float surface_z;

        if (v00 != v00 || v10 != v10 || v01 != v01 || v11 != v11) {
            const int nnx = min(max((int)(px + 0.5f), 0), cols - 1);
            const int nny = min(max((int)(py + 0.5f), 0), rows - 1);
            surface_z = dsm[nny * cols + nnx];
            if (surface_z != surface_z) continue;
        } else {
            const float vmin = fminf(fminf(v00, v10), fminf(v01, v11));
            const float vmax = fmaxf(fmaxf(v00, v10), fmaxf(v01, v11));

            if ((vmax - vmin) > discontinuity_threshold) {
                const int nnx = min(max((int)(px + 0.5f), 0), cols - 1);
                const int nny = min(max((int)(py + 0.5f), 0), rows - 1);
                surface_z = dsm[nny * cols + nnx];
            } else {
                surface_z = v00 * (1.0f - fx) * (1.0f - fy)
                          + v10 * fx          * (1.0f - fy)
                          + v01 * (1.0f - fx) * fy
                          + v11 * fx          * fy;
            }
        }

        if (!hit_primary && surface_z > ray_z_primary) hit_primary = 1;
        if (dual_trace && !hit_secondary && surface_z > ray_z_secondary) hit_secondary = 1;
        if (need_dsm && !hit_dsm && surface_z > ray_z_dsm) hit_dsm = 1;

        if (hit_primary && (!dual_trace || hit_secondary) && (!need_dsm || hit_dsm)) {
            unsigned char value_primary = 0;
            if (hit_primary) value_primary |= 1;
            if (mask_primary) value_primary |= 4;
            if (mask_primary && hit_dsm) value_primary |= 2;
            encoded_out[idx] = value_primary;

            if (dual_trace) {
                unsigned char value_secondary = 0;
                if (hit_secondary) value_secondary |= 1;
                if (mask_secondary) value_secondary |= 4;
                if (mask_secondary && hit_dsm) value_secondary |= 2;
                encoded_out_secondary[idx] = value_secondary;
            }
            return;
        }
    }

    unsigned char value_primary = 0;
    if (hit_primary) value_primary |= 1;
    if (mask_primary) value_primary |= 4;
    if (mask_primary && hit_dsm) value_primary |= 2;
    encoded_out[idx] = value_primary;

    if (dual_trace) {
        unsigned char value_secondary = 0;
        if (hit_secondary) value_secondary |= 1;
        if (mask_secondary) value_secondary |= 4;
        if (mask_secondary && hit_dsm) value_secondary |= 2;
        encoded_out_secondary[idx] = value_secondary;
    }
}
"""

_kernel = cp.RawKernel(_KERNEL_CODE, "shadow_raytrace_dtm_dsm_bits")
print("CUDA kernel compiled.")

CUDA kernel compiled.


## 3. Configuration

**Edit this cell before running the notebook.**

This dataclass controls the entire processing run.

### Most important parameters

- **`input_dtm` / `input_dsm`**  
  Paths to the source rasters.

- **`observer_height_m`**  
  Main analysis height.  
  Example: `1.5` m for pedestrian comfort.

- **`dual_trace`**  
  If `True`, the notebook also computes a second height in the same run.

- **`dual_trace_height_m`**  
  Secondary analysis height.  
  Example: `0.1` m for near-ground shadow studies.

- **`object_mask_epsilon_m`**  
  Tolerance used in the object test `DSM > DTM + h + ε`.  
  A small positive value can reduce false positives caused by vertical noise or tiny DSM/DTM mismatches.

- **`date_selection`**  
  Temporal selection logic for the simulation.

### Practical note

The configuration written here is later saved to disk as a JSON snapshot, so every run remains reproducible.


In [ ]:
@dataclass
class Config:
    # ─── Input / output ───
    input_dtm: str = "insert_dtm_path_here.tif"
    input_dsm: str = "insert_dsm_path_here.tif"
    output_dir: str = "insert_output_dir_here"
    overwrite: bool = False

    # ─── Observer heights ───
    observer_height_m: float = 1.5
    dual_trace: bool = True
    dual_trace_height_m: float = 0.1

    # ─── GPU & Tiling ───
    threads_per_block_x: int = 16
    threads_per_block_y: int = 16
    tile_size: int = 4096
    min_buffer_px: int = 512
    max_buffer_px: int = 6000
    max_feature_height_m: float = 110.0

    # ─── Precision ───
    step_divisor: int = 2
    self_shadow_offset_m: float = -1.0
    object_mask_epsilon_m: float = 0.15
    discontinuity_threshold_m: float = 3.0

    # ─── Geographic / solar ───
    lat: float = 44.4949
    lon: float = 11.3426
    timezone_name: str = "Europe/Rome"

    # ─── Timestamp selection (example) ───
    date_selection: dict = field(default_factory=lambda: {
        "dates": {},
        "rules": [
            {
                "name": "summer_sundays_2025",
                "start_date": "2025-07-01",
                "end_date": "2025-08-31",
                "months": [7, 8],
                "weekdays": ["sunday"],
                "time_start": "05:00",
                "time_end": "22:00",
                "step_minutes": 15,
            },
        ],
    })

    # ─── Legacy fallback ───
    date_start: str = "2025-07-27"
    date_end: str = "2025-07-28"
    time_start: str = "05:00"
    time_end: str = "22:00"
    step_minutes: int = 60

cfg = Config()
print(f"Heights: {cfg.observer_height_m}m (primary), {cfg.dual_trace_height_m}m (secondary)")
print(f"Dual trace: {cfg.dual_trace}")
print(f"Output: {cfg.output_dir}")

Heights: 1.5m (primary), 0.1m (secondary)
Dual trace: True
Output: shadow_outputs_dtm_dsm_0_5_final_august


## 4. Helper Functions

This section contains all non-kernel support logic used by the pipeline.

It includes:

- date and recurrence utilities
- timestamp generation
- solar geometry preparation
- ray parameter computation
- DTM / DSM grid validation
- tiling helpers
- GeoTIFF writing
- main orchestration logic for running the CUDA kernel

These functions are the bridge between the high-level configuration and the low-level GPU computation.


In [4]:
_WEEKDAY_TO_INDEX = {
    "monday": 0,
    "tuesday": 1,
    "wednesday": 2,
    "thursday": 3,
    "friday": 4,
    "saturday": 5,
    "sunday": 6,
}

def parse_date(value: str) -> date:
    return datetime.strptime(value, "%Y-%m-%d").date()


def parse_hhmm(value: str) -> dt_time:
    return datetime.strptime(value, "%H:%M").time()


def iter_day_timestamps(
    day: date,
    time_start: dt_time,
    time_end: dt_time,
    step_minutes: int,
):
    if step_minutes <= 0:
        raise ValueError(f"step_minutes must be > 0, got {step_minutes}.")
    if time_end < time_start:
        raise ValueError(
            f"time_end ({time_end:%H:%M}) cannot be earlier than "
            f"time_start ({time_start:%H:%M}) for {day:%Y-%m-%d}."
        )

    current = datetime.combine(day, time_start)
    end_dt = datetime.combine(day, time_end)
    while current <= end_dt:
        yield current
        current += timedelta(minutes=step_minutes)


def resolve_time_window(
    spec: dict[str, object],
    default_start: str,
    default_end: str,
    default_step: int,
    *,
    context: str,
) -> tuple[dt_time, dt_time, int]:
    time_start_raw = str(spec.get("time_start", default_start))
    time_end_raw = str(spec.get("time_end", default_end))

    try:
        step_minutes = int(spec.get("step_minutes", default_step))
    except (TypeError, ValueError) as exc:
        raise ValueError(f"{context}: step_minutes must be an integer.") from exc

    try:
        time_start = parse_hhmm(time_start_raw)
    except ValueError as exc:
        raise ValueError(
            f"{context}: invalid time_start '{time_start_raw}', expected HH:MM."
        ) from exc

    try:
        time_end = parse_hhmm(time_end_raw)
    except ValueError as exc:
        raise ValueError(
            f"{context}: invalid time_end '{time_end_raw}', expected HH:MM."
        ) from exc

    if step_minutes <= 0:
        raise ValueError(f"{context}: step_minutes must be > 0.")
    if time_end < time_start:
        raise ValueError(
            f"{context}: time_end ({time_end:%H:%M}) cannot be earlier than "
            f"time_start ({time_start:%H:%M})."
        )

    return time_start, time_end, step_minutes


def normalize_months(values: object, *, context: str) -> set[int] | None:
    if values is None:
        return None
    if not isinstance(values, (list, tuple, set)):
        raise TypeError(f"{context}: months must be a list, tuple, or set.")

    months: set[int] = set()
    for value in values:
        try:
            month = int(value)
        except (TypeError, ValueError) as exc:
            raise ValueError(f"{context}: invalid month value '{value}'.") from exc
        if month < 1 or month > 12:
            raise ValueError(f"{context}: month values must be in 1..12.")
        months.add(month)

    return months


def normalize_weekdays(values: object, *, context: str) -> set[int] | None:
    if values is None:
        return None
    if not isinstance(values, (list, tuple, set)):
        raise TypeError(f"{context}: weekdays must be a list, tuple, or set.")

    weekdays: set[int] = set()
    for value in values:
        if isinstance(value, str):
            key = value.strip().lower()
            if key not in _WEEKDAY_TO_INDEX:
                valid_days = ", ".join(_WEEKDAY_TO_INDEX)
                raise ValueError(
                    f"{context}: invalid weekday '{value}'. "
                    f"Use one of: {valid_days}."
                )
            weekdays.add(_WEEKDAY_TO_INDEX[key])
            continue

        try:
            weekday = int(value)
        except (TypeError, ValueError) as exc:
            raise ValueError(f"{context}: invalid weekday value '{value}'.") from exc
        if weekday < 0 or weekday > 6:
            raise ValueError(f"{context}: weekday values must be in 0..6.")
        weekdays.add(weekday)

    return weekdays


def has_custom_date_selection(cfg: Config) -> bool:
    schedule = cfg.date_selection or {}
    dates = schedule.get("dates", {})
    rules = schedule.get("rules", [])
    return bool(dates) or bool(rules)


def describe_timestamp_selection(cfg: Config) -> str:
    if has_custom_date_selection(cfg):
        schedule = cfg.date_selection or {}
        dates = schedule.get("dates", {})
        rules = schedule.get("rules", [])
        return (
            "Schedule dictionary: "
            f"{len(dates)} explicit date(s), {len(rules)} rule(s)"
        )

    return (
        "Legacy range: "
        f"{cfg.date_start} - {cfg.date_end} | "
        f"{cfg.time_start}-{cfg.time_end} every {cfg.step_minutes} min"
    )


def iter_local_timestamps(cfg: Config):
    if has_custom_date_selection(cfg):
        schedule = cfg.date_selection or {}
        dates = schedule.get("dates", {})
        rules = schedule.get("rules", [])

        if not isinstance(dates, dict):
            raise TypeError("date_selection['dates'] must be a dictionary.")
        if not isinstance(rules, list):
            raise TypeError("date_selection['rules'] must be a list.")

        timestamps: set[datetime] = set()

        for day_raw, spec in dates.items():
            context = f"date_selection['dates']['{day_raw}']"
            if spec is None:
                spec = {}
            if not isinstance(spec, dict):
                raise TypeError(f"{context} must be a dictionary.")

            try:
                day = parse_date(str(day_raw))
            except ValueError as exc:
                raise ValueError(
                    f"{context}: invalid date '{day_raw}', expected YYYY-MM-DD."
                ) from exc

            time_start, time_end, step_minutes = resolve_time_window(
                spec,
                cfg.time_start,
                cfg.time_end,
                cfg.step_minutes,
                context=context,
            )
            timestamps.update(
                iter_day_timestamps(day, time_start, time_end, step_minutes)
            )

        for rule_index, rule in enumerate(rules, start=1):
            context = f"date_selection['rules'][{rule_index}]"
            if not isinstance(rule, dict):
                raise TypeError(f"{context} must be a dictionary.")

            start_raw = str(rule.get("start_date", cfg.date_start))
            end_raw = str(rule.get("end_date", cfg.date_end))

            try:
                start_day = parse_date(start_raw)
            except ValueError as exc:
                raise ValueError(
                    f"{context}: invalid start_date '{start_raw}', expected YYYY-MM-DD."
                ) from exc

            try:
                end_day = parse_date(end_raw)
            except ValueError as exc:
                raise ValueError(
                    f"{context}: invalid end_date '{end_raw}', expected YYYY-MM-DD."
                ) from exc

            if end_day < start_day:
                raise ValueError(
                    f"{context}: end_date ({end_raw}) cannot be earlier than "
                    f"start_date ({start_raw})."
                )

            months = normalize_months(rule.get("months"), context=context)
            weekdays = normalize_weekdays(rule.get("weekdays"), context=context)
            time_start, time_end, step_minutes = resolve_time_window(
                rule,
                cfg.time_start,
                cfg.time_end,
                cfg.step_minutes,
                context=context,
            )

            day = start_day
            while day <= end_day:
                if months is not None and day.month not in months:
                    day += timedelta(days=1)
                    continue
                if weekdays is not None and day.weekday() not in weekdays:
                    day += timedelta(days=1)
                    continue

                timestamps.update(
                    iter_day_timestamps(day, time_start, time_end, step_minutes)
                )
                day += timedelta(days=1)

        for ts_local in sorted(timestamps):
            yield ts_local
        return

    d0 = parse_date(cfg.date_start)
    d1 = parse_date(cfg.date_end)
    if d1 < d0:
        raise ValueError(
            f"date_end ({cfg.date_end}) cannot be earlier than date_start ({cfg.date_start})."
        )

    t0 = parse_hhmm(cfg.time_start)
    t1 = parse_hhmm(cfg.time_end)
    day = d0
    while day <= d1:
        yield from iter_day_timestamps(day, t0, t1, cfg.step_minutes)
        day += timedelta(days=1)


# ---------------------------------------------------------------------------
# Ray parameter computation with sub-pixel stepping
# ---------------------------------------------------------------------------

def compute_ray_params(
    alt_deg: float,
    azi_deg: float,
    cell_size_x: float,
    cell_size_y: float,
    step_divisor: int = 2,
) -> tuple[float, float, float]:
    """
    Compute ray-marching direction and vertical rise per sub-step.

    With step_divisor > 1 the ray advances in sub-pixel increments:
      step_divisor=1  → 1.0 pixel per step (original behaviour)
      step_divisor=2  → 0.5 pixel per step (default, higher precision)
    """
    alt_rad = math.radians(alt_deg)
    azi_rad = math.radians(azi_deg)

    raw_dcol = math.sin(azi_rad)
    raw_drow = -math.cos(azi_rad)

    max_comp = max(abs(raw_dcol), abs(raw_drow), 1e-12)
    step_col = raw_dcol / max_comp / step_divisor
    step_row = raw_drow / max_comp / step_divisor

    horiz_dist = math.sqrt(
        (step_col * cell_size_x) ** 2 + (step_row * cell_size_y) ** 2
    )

    z_rise_per_step = math.tan(alt_rad) * horiz_dist

    return step_col, step_row, z_rise_per_step


def compute_dynamic_buffer(
    alt_deg: float,
    cell_size: float,
    base_buffer: int,
    max_feature_height: float = 200.0,
    max_buffer: int = 2000,
) -> int:
    if alt_deg <= 2.0:
        return max_buffer
    shadow_len_m = max_feature_height / math.tan(math.radians(alt_deg))
    shadow_len_px = int(shadow_len_m / cell_size) + 32
    return max(base_buffer, min(shadow_len_px, max_buffer))


def estimate_required_buffer_px(
    alt_deg: float,
    cell_size: float,
    max_feature_height: float,
) -> float:
    if alt_deg <= 0:
        return math.inf
    return (max_feature_height / math.tan(math.radians(alt_deg))) / cell_size + 32.0


def compute_self_shadow_offset(cell_size_x: float, cell_size_y: float) -> float:
    """
    Auto-compute self-shadow offset from cell dimensions.

    Prevents a pixel from falsely shadowing itself on sloped ground.
    Scaled to 10% of average cell size (e.g. 0.05m for 0.5m cells).
    """
    avg_cell = (cell_size_x + cell_size_y) / 2.0
    return max(0.05, avg_cell * 0.10)


def compute_directional_buffer(
    azi_deg: float,
    buffer_px: int,
    min_margin: int = 64,
) -> tuple[int, int, int, int]:
    """
    Compute per-direction buffer based on sun azimuth.

    Shadows are cast AWAY from the sun. The obstructions that cast shadows
    into a tile are on the SUN-FACING side. So we only need buffer on the
    side the sun is coming from.

    Returns (buf_north, buf_south, buf_west, buf_east).
    """
    azi_rad = math.radians(azi_deg)
    # Ray direction toward sun in pixel space
    dx = math.sin(azi_rad)    # positive = east
    dy = -math.cos(azi_rad)   # positive = south (row increases downward)

    # Buffer on the side the sun is coming from (where obstructions are)
    buf_east  = max(min_margin, int(buffer_px * max(0.0,  dx)))
    buf_west  = max(min_margin, int(buffer_px * max(0.0, -dx)))
    buf_south = max(min_margin, int(buffer_px * max(0.0,  dy)))
    buf_north = max(min_margin, int(buffer_px * max(0.0, -dy)))

    return buf_north, buf_south, buf_west, buf_east


# ---------------------------------------------------------------------------
# Grid alignment validation
# ---------------------------------------------------------------------------

def validate_grid_alignment(dtm_path: Path, dsm_path: Path) -> None:
    """Verify DTM and DSM are on the same grid (CRS, size, transform)."""
    with rasterio.open(dtm_path) as dtm_src, rasterio.open(dsm_path) as dsm_src:
        # Check CRS
        if dtm_src.crs != dsm_src.crs:
            raise ValueError(
                f"CRS mismatch: DTM={dtm_src.crs}, DSM={dsm_src.crs}. "
                f"Run dtm_resample.py first to align the DTM to the DSM grid."
            )

        # Check dimensions
        if dtm_src.width != dsm_src.width or dtm_src.height != dsm_src.height:
            raise ValueError(
                f"Grid size mismatch: DTM={dtm_src.width}x{dtm_src.height}, "
                f"DSM={dsm_src.width}x{dsm_src.height}. "
                f"Run dtm_resample.py first to align the DTM to the DSM grid."
            )

        # Check transform (origin and pixel size)
        dt = dtm_src.transform
        st = dsm_src.transform
        if abs(dt.a - st.a) > 1e-6 or abs(dt.e - st.e) > 1e-6:
            raise ValueError(
                f"Pixel size mismatch: DTM=({dt.a:.6f}, {dt.e:.6f}), "
                f"DSM=({st.a:.6f}, {st.e:.6f})."
            )
        origin_tol_x = max(abs(dt.a), abs(st.a)) * 1e-3
        origin_tol_y = max(abs(dt.e), abs(st.e)) * 1e-3
        if abs(dt.c - st.c) > origin_tol_x or abs(dt.f - st.f) > origin_tol_y:
            raise ValueError(
                f"Origin mismatch: DTM=({dt.c:.2f}, {dt.f:.2f}), "
                f"DSM=({st.c:.2f}, {st.f:.2f}). "
                f"Tolerance is {origin_tol_x:.6f} x {origin_tol_y:.6f}."
            )


# ---------------------------------------------------------------------------
# Shadow computation for a single timestamp (tiled)
# ---------------------------------------------------------------------------

def write_encoded_shadow_tif(path: Path, data: np.ndarray, meta: dict, observer_height_m: float) -> None:
    """Write encoded uint8 shadow TIFF with dataset tags describing the bit layout."""
    with rasterio.open(path, "w", **meta) as dest:
        dest.write(data, 1)
        dest.update_tags(
            observer_height_m=float(observer_height_m),
            encoding="value = 1*G_h + 2*S_h + 4*M_h",
            bit_0="G_h = raytrace(DTM + h, DSM)",
            bit_1="S_h = filtered DSM surface shadow = S AND M_h",
            bit_2="M_h = DSM > DTM + h",
            valid_values="0,1,4,5,6,7,255",
            impossible_values="2,3",
            nodata_value="255",
        )


def run_shadow_raytrace(
    dtm_cpu: np.ndarray,
    dsm_cpu: np.ndarray,
    cell_size_x: float,
    cell_size_y: float,
    raster_meta: dict,
    output_tif: Path,
    secondary_output_tif: Path | None,
    ts_local: datetime,
    cfg: Config,
    max_dsm_z_global: float,
) -> tuple[float, float, float] | None:
    """Run encoded DTM+DSM shadow computation with optional dual trace.

    Returns (elapsed_sec, alt_deg, azi_deg), or None if sun is below min altitude.

    Output encoding per requested height h:
      bit 0 (1) -> G_h : ground / pedestrian shadow from raytrace(DTM + h, DSM)
      bit 1 (2) -> S_h : filtered DSM shadow where S_h = S AND M_h
      bit 2 (4) -> M_h : object mask where DSM > DTM + h
    """
    output_tif.parent.mkdir(parents=True, exist_ok=True)
    if secondary_output_tif is not None:
        secondary_output_tif.parent.mkdir(parents=True, exist_ok=True)

    # Solar position (apparent_elevation includes atmospheric refraction)
    times = pd.DatetimeIndex([ts_local], tz=cfg.timezone_name)
    sp = get_solarposition(times, cfg.lat, cfg.lon)
    alt_deg = float(sp["apparent_elevation"].iloc[0])
    azi_deg = float(sp["azimuth"].iloc[0])

    if alt_deg <= 5.0:
        print(f"  Sun too low (alt={alt_deg:.2f}°), skipping.")
        return None

    if output_tif.exists() and not cfg.overwrite:
        if secondary_output_tif is None or secondary_output_tif.exists():
            print(f"  Output exists, skipping: {output_tif.name}")
            return 0.0, alt_deg, azi_deg

    full_rows, full_cols = dtm_cpu.shape

    # Ray parameters with sub-pixel stepping
    step_col, step_row, z_rise = compute_ray_params(
        alt_deg, azi_deg, cell_size_x, cell_size_y, cfg.step_divisor
    )

    # Offsets for self-shadow protection.
    if cfg.self_shadow_offset_m < 0:
        auto_offset = compute_self_shadow_offset(cell_size_x, cell_size_y)
    else:
        auto_offset = cfg.self_shadow_offset_m

    primary_offset = 0.0 if cfg.observer_height_m > auto_offset else auto_offset
    if cfg.dual_trace:
        secondary_offset = 0.0 if cfg.dual_trace_height_m > auto_offset else auto_offset
    else:
        secondary_offset = 0.0
    dsm_surface_offset = auto_offset

    avg_cell = (cell_size_x + cell_size_y) / 2.0
    required_buffer_px = estimate_required_buffer_px(
        alt_deg, avg_cell, cfg.max_feature_height_m
    )
    buffer_px = compute_dynamic_buffer(
        alt_deg, avg_cell, cfg.min_buffer_px,
        cfg.max_feature_height_m, cfg.max_buffer_px,
    )
    buf_n, buf_s, buf_w, buf_e = compute_directional_buffer(azi_deg, buffer_px)
    buffer_is_capped = required_buffer_px > cfg.max_buffer_px

    mode_str = "DUAL" if cfg.dual_trace else "SINGLE"
    print(
        f"  Sun: Alt={alt_deg:.2f}  Azi={azi_deg:.2f} | "
        f"step=1/{cfg.step_divisor}px | buffer={buffer_px}px (N{buf_n}/S{buf_s}/W{buf_w}/E{buf_e}) | "
        f"{mode_str} | ground_offset={primary_offset:.2f}m | dsm_offset={dsm_surface_offset:.2f}m"
    )
    if buffer_is_capped:
        if math.isinf(required_buffer_px):
            required_text = "infinite"
        else:
            required_text = f"{required_buffer_px:.0f}"
        print(
            "  WARNING: required shadow buffer is about "
            f"{required_text}px, but max_buffer_px={cfg.max_buffer_px}. "
            "Tile-edge shadows may be underestimated."
        )

    full_encoded_cpu = np.full((full_rows, full_cols), 255, dtype=np.uint8)
    if cfg.dual_trace:
        full_encoded_secondary_cpu = np.full((full_rows, full_cols), 255, dtype=np.uint8)

    start_total = time.time()
    processed_tiles = 0
    skipped_tiles = 0

    for y in range(0, full_rows, cfg.tile_size):
        for x in range(0, full_cols, cfg.tile_size):
            y0, y1 = y, min(y + cfg.tile_size, full_rows)
            x0, x1 = x, min(x + cfg.tile_size, full_cols)

            if np.isnan(dtm_cpu[y0:y1, x0:x1]).all():
                skipped_tiles += 1
                continue

            processed_tiles += 1

            py0 = max(0, y0 - buf_n)
            py1 = min(full_rows, y1 + buf_s)
            px0 = max(0, x0 - buf_w)
            px1 = min(full_cols, x1 + buf_e)

            padded_dtm = dtm_cpu[py0:py1, px0:px1]
            padded_dsm = dsm_cpu[py0:py1, px0:px1]
            d_dtm = cp.asarray(padded_dtm)
            d_dsm = cp.asarray(padded_dsm)

            p_rows, p_cols = padded_dtm.shape
            d_encoded = cp.zeros((p_rows, p_cols), dtype=cp.uint8)
            if cfg.dual_trace:
                d_encoded_secondary = cp.zeros((p_rows, p_cols), dtype=cp.uint8)
            else:
                d_encoded_secondary = d_encoded

            tile_max_dsm_z = float(np.nanmax(padded_dsm)) if np.any(np.isfinite(padded_dsm)) else max_dsm_z_global
            tile_max_steps = (int(math.sqrt(p_rows**2 + p_cols**2)) + 1) * cfg.step_divisor

            tpb = (cfg.threads_per_block_x, cfg.threads_per_block_y)
            blocks = (
                (p_cols + tpb[0] - 1) // tpb[0],
                (p_rows + tpb[1] - 1) // tpb[1],
            )

            _kernel(
                blocks, tpb,
                (
                    d_dtm, d_dsm, d_encoded, d_encoded_secondary,
                    np.int32(p_rows), np.int32(p_cols),
                    np.float32(step_col), np.float32(step_row),
                    np.float32(z_rise), np.int32(tile_max_steps),
                    np.float32(cfg.observer_height_m),
                    np.float32(primary_offset),
                    np.float32(cfg.dual_trace_height_m if cfg.dual_trace else 0.0),
                    np.float32(secondary_offset),
                    np.float32(dsm_surface_offset),
                    np.float32(cfg.object_mask_epsilon_m),
                    np.float32(tile_max_dsm_z),
                    np.float32(cfg.discontinuity_threshold_m),
                    np.int32(1 if cfg.dual_trace else 0),
                ),
            )
            cp.cuda.Device().synchronize()

            padded_encoded = cp.asnumpy(d_encoded)
            if cfg.dual_trace:
                padded_encoded_secondary = cp.asnumpy(d_encoded_secondary)

            d_dtm = None
            d_dsm = None
            d_encoded = None
            d_encoded_secondary = None
            cp.get_default_memory_pool().free_all_blocks()

            ly0 = y0 - py0
            ly1 = ly0 + (y1 - y0)
            lx0 = x0 - px0
            lx1 = lx0 + (x1 - x0)

            full_encoded_cpu[y0:y1, x0:x1] = padded_encoded[ly0:ly1, lx0:lx1]
            if cfg.dual_trace:
                full_encoded_secondary_cpu[y0:y1, x0:x1] = padded_encoded_secondary[ly0:ly1, lx0:lx1]

    print(
        f"  Tiles: {processed_tiles} processed, "
        f"{skipped_tiles} skipped (NaN). Writing TIF..."
    )

    meta = raster_meta.copy()
    meta.update(dtype="uint8", count=1, nodata=255, compress="deflate", zlevel=3)

    write_encoded_shadow_tif(output_tif, full_encoded_cpu, meta, cfg.observer_height_m)
    if cfg.dual_trace and secondary_output_tif is not None:
        write_encoded_shadow_tif(secondary_output_tif, full_encoded_secondary_cpu, meta, cfg.dual_trace_height_m)

    elapsed = time.time() - start_total
    tif_count = 2 if cfg.dual_trace else 1
    print(f"  Done ({tif_count} TIF{'s' if tif_count > 1 else ''}) | {elapsed:.1f}s")

    return elapsed, alt_deg, azi_deg


# ---------------------------------------------------------------------------
# Output helpers
# ---------------------------------------------------------------------------

def format_height_tag(height_m: float) -> str:
    text_h = f"{height_m:.2f}".rstrip("0").rstrip(".").replace(".", "p")
    return f"h{text_h}m"


def build_output_paths(base_dir: Path, ts_local: datetime, cfg: Config) -> tuple[Path, Path | None]:
    base_name = f"shadow_{ts_local:%Y%m%d_%H%M}"
    primary = base_dir / f"{base_name}_{format_height_tag(cfg.observer_height_m)}.tif"
    if cfg.dual_trace:
        secondary = base_dir / f"{base_name}_{format_height_tag(cfg.dual_trace_height_m)}.tif"
    else:
        secondary = None
    return primary, secondary


def build_manifest_path(base_dir: Path) -> Path:
    return base_dir / "shadow_manifest.csv"


# ---------------------------------------------------------------------------
# Path / manifest helpers
# ---------------------------------------------------------------------------

def get_notebook_dir() -> Path:
    """Best-effort notebook directory detection with safe fallback."""
    if "__file__" in globals():
        return Path(__file__).resolve().parent

    session_name = os.environ.get("JPY_SESSION_NAME")
    if session_name:
        session_path = Path(session_name)
        if session_path.is_absolute() and session_path.exists():
            return session_path.resolve().parent

        candidate = (Path.cwd() / session_path).resolve()
        if candidate.exists():
            return candidate.parent

    return Path.cwd().resolve()


NOTEBOOK_DIR = get_notebook_dir()


def resolve_configured_path(path_str: str, base_dir: Path = NOTEBOOK_DIR) -> Path:
    """Resolve config paths relative to the notebook directory when possible."""
    path = Path(path_str).expanduser()
    if path.is_absolute():
        return path.resolve()
    return (base_dir / path).resolve()


def load_current_run_output_files(manifest_path: Path) -> list[Path]:
    """Return unique existing output files referenced by the current manifest."""
    if not manifest_path.exists():
        raise FileNotFoundError(f"Manifest not found: {manifest_path}")

    manifest_df = pd.read_csv(manifest_path)
    if manifest_df.empty:
        return []

    error_series = manifest_df["error"].fillna("").astype(str).str.strip()
    ok_df = manifest_df.loc[error_series == "", ["output_file"]].dropna()

    files: list[Path] = []
    missing: list[Path] = []

    for raw_path in ok_df["output_file"]:
        path = Path(str(raw_path))
        path = path.resolve() if path.is_absolute() else (manifest_path.parent / path).resolve()
        if path.exists():
            files.append(path)
        else:
            missing.append(path)

    unique_files = list(dict.fromkeys(files))

    if missing:
        print(f"WARNING: {len(missing)} manifest-listed outputs were not found on disk.")

    return unique_files


def load_current_run_manifest(manifest_path: Path) -> pd.DataFrame:
    """Load manifest rows with successful outputs only."""
    if not manifest_path.exists():
        raise FileNotFoundError(f"Manifest not found: {manifest_path}")

    manifest_df = pd.read_csv(manifest_path)
    if manifest_df.empty:
        return manifest_df

    manifest_df = manifest_df.copy()
    manifest_df["error"] = manifest_df["error"].fillna("").astype(str).str.strip()
    manifest_df = manifest_df[manifest_df["error"] == ""].copy()
    if "observer_height_m" in manifest_df.columns:
        manifest_df["observer_height_m"] = pd.to_numeric(
            manifest_df["observer_height_m"], errors="coerce"
        )
    return manifest_df


def select_dual_trace_pair(
    manifest_path: Path,
    primary_height_m: float,
    secondary_height_m: float,
    atol: float = 1e-9,
) -> tuple[pd.Series, pd.Series] | tuple[None, None]:
    """Pick a comparable timestamp pair for primary/secondary heights using the manifest."""
    manifest_df = load_current_run_manifest(manifest_path)
    if manifest_df.empty:
        return None, None

    pairs: list[tuple[float, pd.Series, pd.Series]] = []

    for ts_local, group in manifest_df.groupby("timestamp_local", sort=True):
        primary_group = group[np.isclose(group["observer_height_m"], primary_height_m, atol=atol)]
        secondary_group = group[np.isclose(group["observer_height_m"], secondary_height_m, atol=atol)]
        if primary_group.empty or secondary_group.empty:
            continue

        primary_row = primary_group.sort_values("solar_altitude_deg", ascending=False).iloc[0]
        secondary_row = secondary_group.sort_values("solar_altitude_deg", ascending=False).iloc[0]
        altitude = float(primary_row.get("solar_altitude_deg", 0.0))
        pairs.append((altitude, primary_row, secondary_row))

    if not pairs:
        return None, None

    pairs.sort(key=lambda item: item[0], reverse=True)
    _, primary_row, secondary_row = pairs[0]
    return primary_row, secondary_row


print("All helper functions defined.")

All helper functions defined.


## 5. Load & Validate Input Data

This section reads the DTM and DSM from disk and verifies that they are compatible.

The validation step is important because the whole encoding depends on a **shared grid**:

- same CRS
- same transform
- same width / height
- same resolution
- same spatial alignment

If these conditions are not satisfied, the shadow output is not trustworthy.


In [8]:
input_dtm = resolve_configured_path(cfg.input_dtm)
input_dsm = resolve_configured_path(cfg.input_dsm)
output_dir = resolve_configured_path(cfg.output_dir)
output_dir.mkdir(parents=True, exist_ok=True)
manifest_path = build_manifest_path(output_dir)

print(f"Notebook base directory: {NOTEBOOK_DIR}")
print(f"Resolved DTM: {input_dtm}")
print(f"Resolved DSM: {input_dsm}")
print(f"Resolved output_dir: {output_dir}")

if not input_dtm.exists():
    raise FileNotFoundError(f"DTM not found: {input_dtm}")
if not input_dsm.exists():
    raise FileNotFoundError(f"DSM not found: {input_dsm}")

print("Validating DTM/DSM grid alignment...")
validate_grid_alignment(input_dtm, input_dsm)
print("  Grids aligned OK.")

# Read rasters
print("Reading DTM...")
with rasterio.open(input_dtm) as src:
    dtm_data = src.read(1).astype(np.float32)
    meta = src.meta.copy()
    transform = src.transform
    dtm_nodata = src.nodata
if dtm_nodata is not None:
    dtm_data[dtm_data == dtm_nodata] = np.nan

print("Reading DSM...")
with rasterio.open(input_dsm) as src:
    dsm_data = src.read(1).astype(np.float32)
    dsm_nodata = src.nodata
if dsm_nodata is not None:
    dsm_data[dsm_data == dsm_nodata] = np.nan

cell_size_x = abs(transform.a)
cell_size_y = abs(transform.e)

# Safety checks
if transform.e > 0:
    raise ValueError("Raster not north-up.")
if meta.get("crs") and rasterio.crs.CRS(meta["crs"]).is_geographic:
    raise ValueError("Geographic CRS. Must be projected.")
if abs(transform.b) > 1e-10 or abs(transform.d) > 1e-10:
    raise ValueError("Raster has shear/rotation.")

max_dsm_z = float(np.nanmax(dsm_data))

print(f"DTM: {dtm_data.shape[0]}x{dtm_data.shape[1]}, Z: {np.nanmin(dtm_data):.1f}-{np.nanmax(dtm_data):.1f} m")
print(f"DSM: {dsm_data.shape[0]}x{dsm_data.shape[1]}, Z: {np.nanmin(dsm_data):.1f}-{max_dsm_z:.1f} m")
print(f"Cell: {cell_size_x:.4f} x {cell_size_y:.4f} m")


Notebook base directory: C:\Users\leona\Desktop\TALEA_SHADOW_TESTING\zz_code\notebooks
Resolved DTM: C:\Users\leona\Desktop\TALEA_SHADOW_TESTING\zz_code\notebooks\output\dtm_bologna_0.5x0.5_final.tif
Resolved DSM: C:\Users\leona\Desktop\TALEA_SHADOW_TESTING\zz_code\notebooks\output\dsm_bologna_final.tif
Resolved output_dir: C:\Users\leona\Desktop\TALEA_SHADOW_TESTING\zz_code\notebooks\shadow_outputs_dtm_dsm_0_5_final_august
Validating DTM/DSM grid alignment...
  Grids aligned OK.
Reading DTM...
Reading DSM...
DTM: 30269x32568, Z: 0.0-392.9 m
DSM: 30269x32568, Z: -24.8-402.5 m
Cell: 0.5000 x 0.5000 m


### 5b. Save Input Grid Metadata

This cell saves a lightweight JSON description of the input DTM and DSM grids.

Why this is useful:

- lets you inspect the exact raster geometry used in the run
- makes later debugging much easier
- preserves important metadata even if the original files move or are replaced


In [9]:
with rasterio.open(input_dtm) as src:
    dtm_info = {
        "shape": [src.height, src.width],
        "crs": str(src.crs),
        "transform": list(src.transform)[:6],
        "bounds": list(src.bounds),
        "dtype": src.dtypes[0],
        "nodata": src.nodata,
    }
with rasterio.open(input_dsm) as src:
    dsm_info = {
        "shape": [src.height, src.width],
        "crs": str(src.crs),
        "transform": list(src.transform)[:6],
        "bounds": list(src.bounds),
        "dtype": src.dtypes[0],
        "nodata": src.nodata,
    }

grid_info_path = output_dir / "input_grid_metadata.json"
with open(grid_info_path, "w", encoding="utf-8") as f:
    json.dump({"dtm": dtm_info, "dsm": dsm_info}, f, indent=2)
print(f"Saved: {grid_info_path}")

Saved: C:\Users\leona\Desktop\TALEA_SHADOW_TESTING\zz_code\notebooks\shadow_outputs_dtm_dsm_0_5_final_august\input_grid_metadata.json


## 6. Generate Timestamps & Save Config Snapshot

This section does two things:

1. builds the list of local timestamps to process
2. saves a complete snapshot of the run configuration

This is one of the most important reproducibility blocks in the notebook.

At the end of this step you should know exactly:

- how many timestamps will be processed
- the first and last timestamp
- which parameters controlled the run
- where inputs and outputs are located


In [10]:
timestamps = list(iter_local_timestamps(cfg))
if not timestamps:
    raise RuntimeError("No timestamps generated!")

if cfg.self_shadow_offset_m < 0:
    auto_offset = compute_self_shadow_offset(cell_size_x, cell_size_y)
else:
    auto_offset = cfg.self_shadow_offset_m

print(f"Timestamps: {len(timestamps)}")
print(f"First: {timestamps[0]:%Y-%m-%d %H:%M}")
print(f"Last:  {timestamps[-1]:%Y-%m-%d %H:%M}")
print(f"Self-shadow offset: {auto_offset:.3f} m")

# Config snapshot for reproducibility
run_info = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "input_dtm": str(input_dtm),
    "input_dsm": str(input_dsm),
    "output_dir": str(output_dir),
    "n_timestamps": len(timestamps),
    "first_timestamp": timestamps[0].isoformat(),
    "last_timestamp": timestamps[-1].isoformat(),
    "config": asdict(cfg),
}
config_path = output_dir / "run_config_snapshot.json"
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(run_info, f, indent=2, ensure_ascii=False, default=str)
print(f"Saved config snapshot: {config_path}")

Timestamps: 345
First: 2025-08-03 05:00
Last:  2025-08-31 22:00
Self-shadow offset: 0.050 m
Saved config snapshot: C:\Users\leona\Desktop\TALEA_SHADOW_TESTING\zz_code\notebooks\shadow_outputs_dtm_dsm_0_5_final_august\run_config_snapshot.json


## 7. Run Shadow Extraction

This is the main execution loop.

For each timestamp, the notebook:

1. computes solar position
2. skips timestamps with very low solar altitude
3. prepares the GPU ray parameters
4. runs the tiled CUDA shadow extraction
5. writes one or two encoded GeoTIFFs depending on the configuration
6. records output metadata for the manifest

### Important interpretation

The output written here is the **master encoded raster**.  
It is not yet a final thematic layer such as "pedestrian shade frequency" or "hybrid visualization". Those are extracted later from the encoded bits.


In [11]:
rows_out = []
failures = 0
skipped_night = 0

for idx, ts_local in enumerate(timestamps, start=1):
    primary_out, secondary_out = build_output_paths(output_dir, ts_local, cfg)
    primary_existed = primary_out.exists()
    secondary_existed = secondary_out.exists() if secondary_out is not None else False
    print(f"[{idx}/{len(timestamps)}] {ts_local:%Y-%m-%d %H:%M}")

    try:
        result = run_shadow_raytrace(
            dtm_data, dsm_data, cell_size_x, cell_size_y, meta,
            primary_out, secondary_out, ts_local, cfg, max_dsm_z,
        )
        if result is None:
            skipped_night += 1
            continue
        elapsed_s, alt, azi = result

        rows_out.append({
            "timestamp_local": ts_local.strftime("%Y-%m-%d %H:%M"),
            "solar_altitude_deg": round(alt, 2),
            "solar_azimuth_deg": round(azi, 2),
            "observer_height_m": cfg.observer_height_m,
            "output_role": "primary",
            "output_file": primary_out.name,
            "generated": int(cfg.overwrite or not primary_existed or (cfg.dual_trace and secondary_out is not None and not secondary_existed)),
            "elapsed_sec": round(elapsed_s, 3),
            "encoding": "value = 1*G_h + 2*S_h + 4*M_h",
            "error": "",
        })

        if cfg.dual_trace and secondary_out is not None:
            rows_out.append({
                "timestamp_local": ts_local.strftime("%Y-%m-%d %H:%M"),
                "solar_altitude_deg": round(alt, 2),
                "solar_azimuth_deg": round(azi, 2),
                "observer_height_m": cfg.dual_trace_height_m,
                "output_role": "secondary",
                "output_file": secondary_out.name,
                "generated": int(cfg.overwrite or not primary_existed or not secondary_existed),
                "elapsed_sec": round(elapsed_s, 3),
                "encoding": "value = 1*G_h + 2*S_h + 4*M_h",
                "error": "",
            })
    except Exception as exc:
        failures += 1
        print(f"  ERROR: {exc}")
        rows_out.append({
            "timestamp_local": ts_local.strftime("%Y-%m-%d %H:%M"),
            "solar_altitude_deg": "", "solar_azimuth_deg": "",
            "observer_height_m": cfg.observer_height_m,
            "output_role": "primary",
            "output_file": primary_out.name,
            "generated": 0, "elapsed_sec": "",
            "encoding": "value = 1*G_h + 2*S_h + 4*M_h",
            "error": str(exc),
        })
        if cfg.dual_trace and secondary_out is not None:
            rows_out.append({
                "timestamp_local": ts_local.strftime("%Y-%m-%d %H:%M"),
                "solar_altitude_deg": "", "solar_azimuth_deg": "",
                "observer_height_m": cfg.dual_trace_height_m,
                "output_role": "secondary",
                "output_file": secondary_out.name,
                "generated": 0, "elapsed_sec": "",
                "encoding": "value = 1*G_h + 2*S_h + 4*M_h",
                "error": str(exc),
            })

[1/345] 2025-08-03 05:00
  Sun too low (alt=-10.50°), skipping.
[2/345] 2025-08-03 05:15
  Sun too low (alt=-8.34°), skipping.
[3/345] 2025-08-03 05:30
  Sun too low (alt=-6.11°), skipping.
[4/345] 2025-08-03 05:45
  Sun too low (alt=-3.80°), skipping.
[5/345] 2025-08-03 06:00
  Sun too low (alt=-1.44°), skipping.
[6/345] 2025-08-03 06:15
  Sun too low (alt=1.34°), skipping.
[7/345] 2025-08-03 06:30
  Sun too low (alt=3.66°), skipping.
[8/345] 2025-08-03 06:45
  Sun: Alt=6.10  Azi=71.36 | step=1/2px | buffer=2089px (N667/S64/W64/E1979) | DUAL | ground_offset=0.00m | dsm_offset=0.05m
  Tiles: 50 processed, 14 skipped (NaN). Writing TIF...
  Done (2 TIFs) | 60.8s
[9/345] 2025-08-03 07:00
  Sun: Alt=8.62  Azi=73.89 | step=1/2px | buffer=1483px (N411/S64/W64/E1424) | DUAL | ground_offset=0.00m | dsm_offset=0.05m
  Tiles: 50 processed, 14 skipped (NaN). Writing TIF...
  Done (2 TIFs) | 62.4s
[10/345] 2025-08-03 07:15
  Sun: Alt=11.18  Azi=76.40 | step=1/2px | buffer=1145px (N269/S64/W64/E11

KeyboardInterrupt: 

## 8. Write Manifest

The manifest is the document of your run.

Each row links:

- timestamp
- solar geometry
- observer height
- output file path
- role (`primary` / `secondary`)

This file is essential for downstream aggregation, QA, and web visualization pipelines.


In [19]:
with manifest_path.open("w", newline="", encoding="utf-8") as f:
    fieldnames = [
        "timestamp_local", "solar_altitude_deg", "solar_azimuth_deg",
        "observer_height_m", "output_role", "output_file",
        "generated", "elapsed_sec", "encoding", "error",
    ]
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows_out)

generated_count = sum(int(row["generated"]) for row in rows_out if not row["error"])
reused_count = sum(1 for row in rows_out if not row["error"] and not row["generated"])
print(f"Generated: {generated_count}, Reused: {reused_count}, "
      f"Skipped (night): {skipped_night}, Failed: {failures}")
print(f"Manifest: {manifest_path}")

Generated: 4, Reused: 0, Skipped (night): 7, Failed: 0
Manifest: shadow_outputs_dtm_dsm_0_5_final_august\manifest.csv


## 9. QA — Full Integrity Check on Current Run Outputs

This block performs a complete integrity scan on every GeoTIFF listed in the **current run manifest**.

It verifies that:

- only valid values are present
- impossible values `2` and `3` never appear
- nodata is preserved correctly
- a per-file count table is written to disk

Using the manifest avoids mixing the current run with older rasters that may already exist in the same output folder.


In [13]:
allowed_values = {0, 1, 4, 5, 6, 7, 255}
impossible_values = {2, 3}

current_run_tifs = load_current_run_output_files(manifest_path)
if not current_run_tifs:
    raise RuntimeError("No current-run output TIFs found in the manifest for QA.")

qa_rows = []
qa_failed = False

for tif in current_run_tifs:
    with rasterio.open(tif) as src:
        data = src.read(1)

    unique_vals, counts = np.unique(data, return_counts=True)
    value_counts = dict(zip(unique_vals.tolist(), counts.tolist()))

    found_invalid = sorted(set(unique_vals.tolist()) - allowed_values)
    found_impossible = sorted(set(unique_vals.tolist()) & impossible_values)

    qa_rows.append({
        "file": tif.name,
        "path": str(tif),
        "count_0": value_counts.get(0, 0),
        "count_1": value_counts.get(1, 0),
        "count_4": value_counts.get(4, 0),
        "count_5": value_counts.get(5, 0),
        "count_6": value_counts.get(6, 0),
        "count_7": value_counts.get(7, 0),
        "count_255": value_counts.get(255, 0),
        "invalid_values": str(found_invalid) if found_invalid else "",
        "impossible_values": str(found_impossible) if found_impossible else "",
    })

    if found_invalid or found_impossible:
        qa_failed = True
        print(f"QA ERROR: {tif.name}")
        if found_invalid:
            print(f"  Invalid: {found_invalid}")
        if found_impossible:
            print(f"  Impossible: {found_impossible}")

qa_csv = output_dir / "qa_value_distribution.csv"
with open(qa_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=qa_rows[0].keys())
    writer.writeheader()
    writer.writerows(qa_rows)

print(f"\nQA table: {qa_csv}")
print(f"Files checked (current run): {len(current_run_tifs)}")

if qa_failed:
    raise RuntimeError("QA FAILED: invalid or impossible pixel values found!")
else:
    print("QA PASSED: only valid values in all current-run TIFs.")



QA table: C:\Users\leona\Desktop\TALEA_SHADOW_TESTING\zz_code\notebooks\shadow_outputs_dtm_dsm_0_5_final_august\qa_value_distribution.csv
Files checked (current run): 4
QA PASSED: only valid values in all current-run TIFs.


## 10. Statistical Report

This block aggregates value counts across the GeoTIFFs referenced by the **current run QA table**.

It is not a scientific result by itself; it is a **sanity check**.

Use it to detect issues such as:

- unexpectedly large object masks
- almost no DSM surface shadow
- suspiciously high nodata counts
- distributions that look implausible for the study area

If the totals are clearly unrealistic, the run should be inspected before moving to aggregation.


In [14]:
qa_df = pd.read_csv(output_dir / "qa_value_distribution.csv")

summary = {
    "n_files": len(qa_df),
    "sum_0": int(qa_df["count_0"].sum()),
    "sum_1": int(qa_df["count_1"].sum()),
    "sum_4": int(qa_df["count_4"].sum()),
    "sum_5": int(qa_df["count_5"].sum()),
    "sum_6": int(qa_df["count_6"].sum()),
    "sum_7": int(qa_df["count_7"].sum()),
    "sum_255": int(qa_df["count_255"].sum()),
}

summary_csv = output_dir / "qa_summary_totals.csv"
pd.DataFrame([summary]).to_csv(summary_csv, index=False)

# Derived bit totals
G_total = summary["sum_1"] + summary["sum_5"] + summary["sum_7"]
S_total = summary["sum_6"] + summary["sum_7"]
M_total = summary["sum_4"] + summary["sum_5"] + summary["sum_6"] + summary["sum_7"]
valid_total = sum(v for k, v in summary.items() if k.startswith("sum_") and k != "sum_255")

print("=== Pixel Value Totals (current run) ===")
for k, v in summary.items():
    if k != "n_files":
        print(f"  {k}: {v:>15,}")

print(f"\n=== Derived Bit Totals ===")
print(f"  G_h (ground shadow):   {G_total:>15,}  ({G_total/max(valid_total,1)*100:.1f}%)")
print(f"  S_h (surface shadow):  {S_total:>15,}  ({S_total/max(valid_total,1)*100:.1f}%)")
print(f"  M_h (object mask):     {M_total:>15,}  ({M_total/max(valid_total,1)*100:.1f}%)")
print(f"\nSaved: {summary_csv}")


=== Pixel Value Totals (current run) ===
  sum_0:     455,241,704
  sum_1:     866,936,378
  sum_4:       2,868,704
  sum_5:     186,864,619
  sum_6:               0
  sum_7:     744,715,943
  sum_255:   1,686,575,820

=== Derived Bit Totals ===
  G_h (ground shadow):     1,798,516,940  (79.7%)
  S_h (surface shadow):      744,715,943  (33.0%)
  M_h (object mask):         934,449,266  (41.4%)

Saved: C:\Users\leona\Desktop\TALEA_SHADOW_TESTING\zz_code\notebooks\shadow_outputs_dtm_dsm_0_5_final_august\qa_summary_totals.csv


## 11. Cross-Height Comparison (primary vs secondary height)

This diagnostic compares one matched timestamp for the two analysis heights using the **manifest**, so the two rasters are guaranteed to refer to the same timestamp.

It reports three different views:

1. **`G_h` only**  
   Pure ground / pedestrian shadow.

2. **`M_h` only**  
   Pixels already occupied by an object above the analysis height.

3. **`G_h OR M_h`**  
   A combined "not sunlit at height h" footprint. This can be useful, but it must **not** be labeled as pure shadow.

Keeping these metrics separate avoids semantic confusion in the final diagnostic.


In [21]:
manifest_path = Path(cfg.output_dir) / "manifest.csv"

primary_row, secondary_row = select_dual_trace_pair(
    manifest_path,
    cfg.observer_height_m,
    cfg.dual_trace_height_m,
)

if primary_row is not None and secondary_row is not None:
    sample_h15 = (Path(cfg.output_dir) / Path(str(primary_row["output_file"])).name).resolve()
    sample_h01 = (Path(cfg.output_dir) / Path(str(secondary_row["output_file"])).name).resolve()

    with rasterio.open(sample_h15) as src:
        a = src.read(1)
    with rasterio.open(sample_h01) as src:
        b = src.read(1)

    valid = (a != 255) & (b != 255)
    valid_count = int(valid.sum())
    if valid_count == 0:
        raise RuntimeError("No overlapping valid pixels found for cross-height comparison.")

    # Pure ground / pedestrian shadow = G_h (bit 0)
    ground_shadow_a = ((a & 1) != 0) & valid
    ground_shadow_b = ((b & 1) != 0) & valid

    # Object mask = M_h (bit 2)
    mask_a = ((a & 4) != 0) & valid
    mask_b = ((b & 4) != 0) & valid

    # Combined footprint: either shadowed at height h or already occupied by an object above h
    occupied_or_shadow_a = (ground_shadow_a | mask_a) & valid
    occupied_or_shadow_b = (ground_shadow_b | mask_b) & valid

    same_ground = (ground_shadow_a == ground_shadow_b) & valid
    diff_ground = (ground_shadow_a != ground_shadow_b) & valid

    print(f"Matched timestamp: {primary_row['timestamp_local']}")
    print(f"Primary ({cfg.observer_height_m}m):   {sample_h15.name}")
    print(f"Secondary ({cfg.dual_trace_height_m}m): {sample_h01.name}")
    print(f"Solar altitude: {float(primary_row['solar_altitude_deg']):.2f}°")
    print(f"\nValid pixels: {valid_count:,}")

    print(f"\n=== Ground / pedestrian shadow only (G_h, bit 0) ===")
    print(f"  G_h at {cfg.observer_height_m}m: {int(ground_shadow_a.sum()):,} ({ground_shadow_a.sum()/valid_count*100:.1f}%)")
    print(f"  G_h at {cfg.dual_trace_height_m}m: {int(ground_shadow_b.sum()):,} ({ground_shadow_b.sum()/valid_count*100:.1f}%)")
    print(f"  Same state:   {int(same_ground.sum()):,}")
    print(f"  Different:    {int(diff_ground.sum()):,} ({diff_ground.sum()/valid_count*100:.4f}%)")

    print(f"\n=== Object mask only (M_h, bit 2) ===")
    print(f"  M_h at {cfg.observer_height_m}m: {int(mask_a.sum()):,} ({mask_a.sum()/valid_count*100:.1f}%)")
    print(f"  M_h at {cfg.dual_trace_height_m}m: {int(mask_b.sum()):,} ({mask_b.sum()/valid_count*100:.1f}%)")
    print(f"  M_h difference: {int(mask_b.sum() - mask_a.sum()):,} more pixels at {cfg.dual_trace_height_m}m")

    print(f"\n=== Combined occupied-or-shadow footprint (G_h OR M_h) ===")
    print(f"  At {cfg.observer_height_m}m: {int(occupied_or_shadow_a.sum()):,} ({occupied_or_shadow_a.sum()/valid_count*100:.1f}%)")
    print(f"  At {cfg.dual_trace_height_m}m: {int(occupied_or_shadow_b.sum()):,} ({occupied_or_shadow_b.sum()/valid_count*100:.1f}%)")

    # Sanity: lower analysis height should never reduce the object mask.
    violates = int((mask_a & ~mask_b & valid).sum())
    if violates > 0:
        print(f"\nWARNING: {violates} pixels where M_h({cfg.observer_height_m}m)=1 but M_h({cfg.dual_trace_height_m}m)=0 — should not happen!")
    else:
        print(f"\nSanity check: M_h({cfg.dual_trace_height_m}m) >= M_h({cfg.observer_height_m}m) everywhere. OK.")
else:
    print("Dual-trace comparison unavailable: no matched primary/secondary pair found in the current manifest.")


Matched timestamp: 2025-08-03 07:00
Primary (1.5m):   shadow_20250803_0700_h1p5m.tif
Secondary (0.1m): shadow_20250803_0700_h0p1m.tif
Solar altitude: 8.62°

Valid pixels: 564,156,837

=== Ground / pedestrian shadow only (G_h, bit 0) ===
  G_h at 1.5m: 407,761,495 (72.3%)
  G_h at 0.1m: 467,337,829 (82.8%)
  Same state:   504,580,503
  Different:    59,576,334 (10.5602%)

=== Object mask only (M_h, bit 2) ===
  M_h at 1.5m: 201,469,828 (35.7%)
  M_h at 0.1m: 265,754,805 (47.1%)
  M_h difference: 64,284,977 more pixels at 0.1m

=== Combined occupied-or-shadow footprint (G_h OR M_h) ===
  At 1.5m: 408,942,430 (72.5%)
  At 0.1m: 467,844,701 (82.9%)

Sanity check: M_h(0.1m) >= M_h(1.5m) everywhere. OK.


## Done

All outputs are written to the `output_dir` folder.

### Main files produced

- `shadow_YYYYMMDD_HHMM_hXpYm.tif`  
  Per-timestamp 3-bit encoded shadow raster

- `shadow_manifest.csv`  
  Complete inventory of all generated outputs

- `run_config_snapshot.json`  
  Exact parameters used for the run

- `input_grid_metadata.json`  
  Geometry and metadata of the input DTM / DSM

- `qa_value_distribution.csv`  
  Per-file counts of encoded values

- `qa_summary_totals.csv`  
  Aggregate value totals across the run

### What to do next

From these master rasters you can derive:

- total pedestrian shade
- open-ground-only products
- DSM-above-object products
- hybrid visualization layers
- temporal frequencies and sun-hours in downstream aggregation scripts

For more information on Aggregation and Analysis of the data check the other notebooks and documentations.
